In [ ]:
import numpy as np
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
from netCDF4 import Dataset
import matplotlib.pyplot as plt
import pickle

import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return start

here = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
repo_root = find_repo_root(here)
sys.path.append(str(repo_root / 'common' / 'python' / 'io'))
sys.path.append(str(repo_root / 'projects' / 'matlab2python' / 'shared' / 'python'))
sys.path.append(str(repo_root / 'common' / 'python' / 'plotting'))
sys.path.append('../util/shared/python/')
sys.path.append(str(repo_root / 'projects' / 'discover_JH'))
sys.path.append(str(repo_root / 'projects' / 'utils' / 'scripts'))

EASE_PATH = repo_root / 'common' / 'python' / 'plotting' / 'ease_grids'

from read_GEOSldas          import read_tilecoord, read_obs_param

from mapper_functions import plot_aus_tight_pcm, plot_global_tight_pcm

In [ ]:
species_groups = {
    "SMOS": [0, 1, 2, 3],
    "SMAP": [4, 5, 6, 7],
    "ASCAT": [8, 9, 10],
    "CYGN": [13],
    "MODIS": [11, 12]
}

In [ ]:
# Read in the OL data files

stats_file_OL = '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/temporal_stats_OL_20000601_20240531.nc4'

print('reading stats nc4 file '+stats_file_OL)
stats_OL = {}
with Dataset(stats_file_OL,'r') as nc:
    for key, value in nc.variables.items():
        print(f"Reading variable: {key}")
        stats_OL[key] = value[:].filled(np.nan)

ts_stats_file_OL = '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/spatial_stats_OL_200006_202405.pkl'

with open(ts_stats_file_OL, 'rb') as f:
    loaded_data = pickle.load(f)
stats_dict_OL = loaded_data
date_vec_OL = loaded_data.get('date_vec', None)  
date_vec = date_vec_OL 

In [ ]:
# Read in the DA data files
stats_file_DA = '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/temporal_stats_DA_20000601_20240531.nc4'

print('reading stats nc4 file '+stats_file_DA)
stats_DA = {}
with Dataset(stats_file_DA,'r') as nc:
    for key, value in nc.variables.items():
        stats_DA[key] = value[:].filled(np.nan)

ts_stats_file_DA = '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/spatial_stats_DA_200006_202405.pkl'

with open(ts_stats_file_DA, 'rb') as f:
    loaded_data = pickle.load(f)
stats_dict_DA = loaded_data
date_vec_DA = loaded_data.get('date_vec', None) 

In [ ]:
# Sample of final compuation of selected diagnostic metrics for OL
 
Nmin = 20

# Then computer metrics of O-F, O-A, etc. based on above computed
N_data = stats_OL['N_data']
O_mean = stats_OL['O_mean']
A_mean = stats_OL['A_mean']
F_mean = stats_OL['F_mean']
O_stdv = stats_OL['O_stdv']
A_stdv = stats_OL['A_stdv']
F_stdv = stats_OL['F_stdv']
OmF_mean = stats_OL['OmF_mean']
OmF_stdv = stats_OL['OmF_stdv']
OmF_norm_mean = stats_OL['OmF_norm_mean']
OmF_norm_stdv = stats_OL['OmF_norm_stdv']
OmA_mean = stats_OL['OmA_mean']
OmA_stdv = stats_OL['OmA_stdv']
  
# Mask out data points with insufficent observations using the Nmin threshold
# Do NOT apply to N_data
OmF_mean[     N_data < Nmin] = np.nan
OmF_stdv[     N_data < Nmin] = np.nan
OmF_norm_mean[N_data < Nmin] = np.nan
OmF_norm_stdv[N_data < Nmin] = np.nan
OmA_mean[     N_data < Nmin] = np.nan
OmA_stdv[     N_data < Nmin] = np.nan
N_data[       N_data < Nmin] = 0

OmF_mean_OL = OmF_mean
OmF_stdv_OL = OmF_stdv
OmF_norm_mean_OL = OmF_norm_mean
OmF_norm_stdv_OL = OmF_norm_stdv
OmA_mean_OL = OmA_mean
OmA_stdv_OL = OmA_stdv
N_data_OL = N_data

group_metrics_OL = {}

for group, species_indices in species_groups.items():
    group_metrics_OL[group] = {}
    group_N_data = np.nansum(N_data[:, species_indices], axis=1)
    
    group_metrics_OL[group]['OmF_mean'] = np.nansum(OmF_mean[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_OL[group]['OmF_stdv'] = np.nansum(OmF_stdv[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_OL[group]['OmF_norm_mean'] = np.nansum(OmF_norm_mean[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_OL[group]['OmF_norm_stdv'] = np.nansum(OmF_norm_stdv[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_OL[group]['OmA_mean'] = np.nansum(OmA_mean[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_OL[group]['OmA_stdv'] = np.nansum(OmA_stdv[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_OL[group]['Nobs_data'] = group_N_data


In [ ]:

# Sample of final compuation of selected diagnostic metrics for DA

# Then computer metrics of O-F, O-A, etc. based on above computed
N_data = stats_DA['N_data']
O_mean = stats_DA['O_mean']
A_mean = stats_DA['A_mean']
F_mean = stats_DA['F_mean']
O_stdv = stats_DA['O_stdv']
A_stdv = stats_DA['A_stdv']
F_stdv = stats_DA['F_stdv']
OmF_mean = stats_DA['OmF_mean']
OmF_stdv = stats_DA['OmF_stdv']
OmF_norm_mean = stats_DA['OmF_norm_mean']
OmF_norm_stdv = stats_DA['OmF_norm_stdv']
OmA_mean = stats_DA['OmA_mean']
OmA_stdv = stats_DA['OmA_stdv']

# Mask out data points with insufficent observations using the Nmin threshold
# Do NOT apply to N_data
OmF_mean[     N_data < Nmin] = np.nan
OmF_stdv[     N_data < Nmin] = np.nan
OmF_norm_mean[N_data < Nmin] = np.nan
OmF_norm_stdv[N_data < Nmin] = np.nan
OmA_mean[     N_data < Nmin] = np.nan
OmA_stdv[     N_data < Nmin] = np.nan
N_data[       N_data < Nmin] = 0
OmF_mean_DA = OmF_mean
OmF_stdv_DA = OmF_stdv
OmF_norm_mean_DA = OmF_norm_mean
OmF_norm_stdv_DA = OmF_norm_stdv
OmA_mean_DA = OmA_mean
OmA_stdv_DA = OmA_stdv
N_data_DA = N_data

group_metrics_DA = {}

for group, species_indices in species_groups.items():
    group_metrics_DA[group] = {}
    group_N_data = np.nansum(N_data[:, species_indices], axis=1)
    
    group_metrics_DA[group]['OmF_mean'] = np.nansum(OmF_mean[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_DA[group]['OmF_stdv'] = np.nansum(OmF_stdv[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_DA[group]['OmF_norm_mean'] = np.nansum(OmF_norm_mean[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_DA[group]['OmF_norm_stdv'] = np.nansum(OmF_norm_stdv[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_DA[group]['OmA_mean'] = np.nansum(OmA_mean[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_DA[group]['OmA_stdv'] = np.nansum(OmA_stdv[:, species_indices] * N_data[:, species_indices], axis=1) / group_N_data
    group_metrics_DA[group]['Nobs_data'] = group_N_data


In [ ]:
ftc = '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/LS_OLv8_M36.ldas_tilecoord.bin'
tc = read_tilecoord(ftc)
n_tile = tc['N_tile']
lat = tc['com_lat']
lon = tc['com_lon']

map_array = np.empty([n_tile, 3])
map_array.fill(np.nan)
map_array[:, 1] = lon
map_array[:, 2] = lat

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Number of days used for mean obs/day for each observing system
obs_days_by_group = {
    'ASCAT': 6121,
    'SMOS': 5047,
    'SMAP': 3287,
    'MODIS': 8700,
    'CYGN': 2131,
}

panel_order = [
    ('MODIS', 'MODIS'),
    ('ASCAT', 'ASCAT'),
    ('SMOS', 'SMOS'),
    ('SMAP', 'SMAP'),
    ('CYGN', 'CYGNSS'),
]
panel_labels = ['a', 'b', 'c', 'd', 'e']

# EASE2 M36 global grid (used by existing map functions)
lats2d = np.fromfile(str(EASE_PATH / 'EASE2_M36km.lats.964x406x1.double'), dtype=np.float64).reshape((406, 964))
lons2d = np.fromfile(str(EASE_PATH / 'EASE2_M36km.lons.964x406x1.double'), dtype=np.float64).reshape((406, 964))
lats_row = lats2d[:, 1]
lons_col = lons2d[1, :]

# Precompute tile -> grid index once
row_idx = np.array([np.abs(lats_row - la).argmin() for la in lat], dtype=int)
col_idx = np.array([np.abs(lons_col - lo).argmin() for lo in lon], dtype=int)

fig, axs = plt.subplots(
    3,
    2,
    figsize=(14, 14),
    subplot_kw={'projection': ccrs.Robinson()},
    constrained_layout=True,
)
axs = np.array(axs).reshape(3, 2)

mappable = None
for i, (group, display_name) in enumerate(panel_order):
    row, col = divmod(i, 2)
    ax = axs[row, col]

    n_days = obs_days_by_group[group]
    obs_per_day = np.array(group_metrics_DA[group]['Nobs_data'], dtype=float) / float(n_days)

    # Build gridded field for pcolormesh
    grid = np.full(lats2d.shape, np.nan, dtype=float)
    valid = np.isfinite(obs_per_day)
    grid[row_idx[valid], col_idx[valid]] = obs_per_day[valid]

    # Exclude Antarctica
    grid = np.where(lats2d < -60.0, np.nan, grid)

    spatial_mean = np.nanmean(grid)
    spatial_std = np.nanstd(grid)

    mappable = ax.pcolormesh(
        lons2d,
        lats2d,
        grid,
        transform=ccrs.PlateCarree(),
        cmap='viridis',
        vmin=0,
        vmax=2,
        shading='auto',
    )
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
    ax.coastlines(linewidth=0.5)
    ax.set_global()
    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    ax.set_title(f"{panel_labels[i]}) {display_name}: Obs/day (mean over {n_days} days)", fontsize=11)
    ax.text(
        0.02,
        0.03,
        f"Spatial mean: {spatial_mean:.3f} +/- {spatial_std:.3f}",
        transform=ax.transAxes,
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'),
    )

# Keep 3x2 layout and hide the unused panel
axs[2, 1].axis('off')

cbar = fig.colorbar(
    mappable,
    ax=axs,
    orientation='horizontal',
    fraction=0.04,
    pad=0.04,
)
cbar.set_label('Number of observations per day')
cbar.set_ticks(np.linspace(0, 2, 5))

plt.show()

In [ ]:
def convert_stats_dict_to_arrays(stats_dict):
    """Convert dictionary of lists to numpy arrays"""
    array_dict = {}
    
    for key in stats_dict.keys():
        # Convert list to array and reshape
        array_dict[key] = np.array(stats_dict[key])
        
        # Check if we need to handle missing values (-- in data)
        if isinstance(array_dict[key][0], (list, np.ndarray)):
            # Replace '--' with np.nan
            temp_array = []
            for row in array_dict[key]:
                cleaned_row = [np.nan if x == '--' else float(x) for x in row]
                temp_array.append(cleaned_row)
            array_dict[key] = np.array(temp_array)
    
    return array_dict

# Convert dictionary
stats_dict_DA_arrays = convert_stats_dict_to_arrays(stats_dict_DA)
stats_dict_OL_arrays = convert_stats_dict_to_arrays(stats_dict_OL)

# Convert date vector to datetime objects
date_vec_DA = [datetime.strptime(date, '%Y%m') for date in date_vec_DA]
date_vec_OL = [datetime.strptime(date, '%Y%m') for date in date_vec_OL]

# Print first few dates to verify
print("Sample dates:", date_vec_DA[:3])
# Print the last few dates to verify
print("Sample dates:", date_vec_DA[-3:])

In [ ]:
def calculate_weighted_group_stats(stats_dict, species_groups):
    """Calculate weighted statistics for each group"""
    
    n_times = len(stats_dict['OmF_mean'])
    stats = ['O_mean','F_mean','OmF_mean', 'OmF_stdv', 'OmA_mean', 'OmA_stdv']
    
    # Initialize output dictionary
    group_stats = {}
    for group in species_groups.keys():
        group_stats[group] = {stat: np.zeros(n_times) for stat in stats}
        group_stats[group]['N_data'] = np.zeros(n_times)
    
    # Calculate weighted stats for each timestep
    for t in range(n_times):
        for group, indices in species_groups.items():
            # Get weights for this group/time
            weights = stats_dict['N_data'][t, indices]
            total_weight = np.sum(weights)
            
            if total_weight > 0:
                # Calculate weighted statistics
                for stat in stats:
                    values = stats_dict[stat][t, indices]
                    group_stats[group][stat][t] = np.average(values, weights=weights)
                group_stats[group]['N_data'][t] = total_weight
            else:
                # Set to NaN if no observations
                for stat in stats:
                    group_stats[group][stat][t] = np.nan
                    
    return group_stats

# Calculate group means
group_ts_DA = calculate_weighted_group_stats(stats_dict_DA_arrays, species_groups)
group_ts_OL = calculate_weighted_group_stats(stats_dict_OL_arrays, species_groups)

In [ ]:
print("length of date_vec_DA", len(date_vec_DA))
print("length of date_vec", len(date_vec))

In [ ]:
targets = ["MODIS", "ASCAT", "SMOS", "SMAP", "CYGN"]
display_names = {"CYGN": "CYGNSS"}
panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)"]

fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)
axes = axes.flatten()

for ax, group, label in zip(axes[:len(targets)], targets, panel_labels):
    o_mean = group_ts_OL[group]["O_mean"]
    f_mean = group_ts_OL[group]["F_mean"]
    mean_o = np.nanmean(o_mean)
    mean_f = np.nanmean(f_mean)

    ax.plot(date_vec_OL, o_mean, "--", label=f"O_mean ({mean_o:.3f})")
    ax.plot(date_vec_OL, f_mean, "-", label=f"F_mean ({mean_f:.3f})")
    name = display_names.get(group, group)
    ax.set_title(f"{name}: LS_OLv8_M36_200006–202405", fontsize=11)
    ax.set_ylabel("O & F Mean")
    ax.text(0.02, 0.92, label, transform=ax.transAxes)
    ax.legend(fontsize=9)

for ax in axes[:4]:
    ax.tick_params(labelbottom=False)

for ax in axes[4:len(targets)]:
    ax.set_xlabel("Date")
    ax.set_xticks(date_vec_OL[::24])
    ax.tick_params(axis="x", rotation=45)

# Keep 3x2 layout and hide the unused panel
axes[-1].axis("off")

plt.tight_layout(h_pad=0.3, w_pad=0.25)
plt.show()

In [ ]:
targets = ["MODIS", "ASCAT", "SMOS", "SMAP", "CYGN"]
display_names = {"CYGN": "CYGNSS"}
panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)"]

fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)
axes = axes.flatten()

for ax, group, label in zip(axes[:len(targets)], targets, panel_labels):
    ol_series = group_ts_OL[group]["OmF_mean"]
    da_series = group_ts_DA[group]["OmF_mean"]
    mean_ol = np.nanmean(ol_series)
    mean_da = np.nanmean(da_series)

    ax.plot(date_vec_OL, ol_series, "--", label=f"OL (mean {mean_ol:.3f})")
    ax.plot(date_vec_DA, da_series, "-", label=f"DA (mean {mean_da:.3f})")
    ax.axhline(0, color="black", linestyle=":", linewidth=1)

    name = display_names.get(group, group)
    ax.set_title(f"{name}: LS_DAv8_M36_200006–202405", fontsize=11)
    ax.set_ylabel("O − F Mean")
    ax.text(0.02, 0.92, label, transform=ax.transAxes)
    ax.legend(fontsize=9)

for ax in axes[:4]:
    ax.tick_params(labelbottom=False)

for ax in axes[4:len(targets)]:
    ax.set_xlabel("Date")
    ax.set_xticks(date_vec_DA[::24])
    ax.tick_params(axis="x", rotation=45)

# Keep 3x2 layout and hide the unused panel
axes[-1].axis("off")

plt.tight_layout(h_pad=0.3, w_pad=0.25)
plt.show()

In [ ]:
targets = ["MODIS", "ASCAT", "SMOS", "SMAP", "CYGN"]
display_names = {"CYGN": "CYGNSS"}
panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)"]

fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)
axes = axes.flatten()

for ax, group, label in zip(axes[:len(targets)], targets, panel_labels):
    ol_series = group_ts_OL[group]["OmF_stdv"]
    da_series = group_ts_DA[group]["OmF_stdv"]
    mean_ol = np.nanmean(ol_series)
    mean_da = np.nanmean(da_series)

    ax.plot(date_vec_DA, ol_series, "--", label=f"OL (mean {mean_ol:.3f})")
    ax.plot(date_vec_DA, da_series, "-", label=f"DA (mean {mean_da:.3f})")

    name = display_names.get(group, group)
    ax.set_title(f"{name}: LS_DAv8_M36_200006–202405", fontsize=11)
    ax.set_ylabel("O − F StdDev")
    ax.text(0.02, 0.92, label, transform=ax.transAxes)
    ax.legend(fontsize=9)

for ax in axes[:4]:
    ax.tick_params(labelbottom=False)

for ax in axes[4:len(targets)]:
    ax.set_xlabel("Date")
    ax.set_xticks(date_vec_DA[::24])
    ax.tick_params(axis="x", rotation=45)

# Keep 3x2 layout and hide the unused panel
axes[-1].axis("off")

plt.tight_layout(h_pad=0.3, w_pad=0.25)
plt.show()

In [ ]:
# Replace zeros with NaNs
for group in species_groups:
    data = group_ts_DA[group]['N_data']
    group_ts_DA[group]['N_data'] = np.where(data == 0, np.nan, data)

# Stack all groups (rows = groups, cols = months)
group_stack = np.vstack([group_ts_DA[g]['N_data'] for g in species_groups])

# Total obs per month (ignores NaNs)
total_obs = np.nansum(group_stack, axis=0)

fig, (ax_top, ax_bottom) = plt.subplots(
    2, 1, figsize=(11, 7), sharex=True, height_ratios=[1.1, 1]
)

# Top panel: total obs / month
ax_top.plot(date_vec_DA, total_obs, color='k', lw=1)
ax_top.set_ylabel('Total Obs / Month')
ax_top.set_title('Number of Observations (DA): LS_DAv8_M36_200006–202405')
ax_top.tick_params(labelbottom=False)  # suppress x ticklabels here
ax_top.text(0.01, 0.92, '(a)', transform=ax_top.transAxes, fontsize=11)

# Bottom panel: per-group series
for group in species_groups:
    ax_bottom.plot(date_vec_DA, group_ts_DA[group]['N_data'], label=group)

ax_bottom.set_ylabel('Obs / Month')
ax_bottom.set_xlabel('Date')
ax_bottom.text(0.01, 0.92, '(b)', transform=ax_bottom.transAxes, fontsize=11)
ax_bottom.legend(ncol=2, fontsize=9)

# x ticks every ~24 months (adjust as you like)
ax_bottom.set_xticks(date_vec_DA[::24])
ax_bottom.tick_params(axis='x', rotation=45)

plt.tight_layout(h_pad=0.2)
plt.show()

In [ ]:
print(len(date_vec_DA))
print(len(group_ts_DA['SMAP']['N_data']))
print(len(group_ts_OL[group]['OmF_stdv']))
print(len(group_ts_DA[group]['OmF_stdv']))


group = 'MODIS'
# Print the last 5 value of these arrays to verify
print("Last 5 dates:", date_vec_DA[-5:])
print("Last 5 OL OmF_stdv:", group_ts_OL[group]['OmF_stdv'][-5:])
print("Last 5 DA OmF_stdv:", group_ts_DA[group]['OmF_stdv'][-5:])
print("Last 5 DA N_data:", group_ts_DA[group]['N_data'][-5:])

In [ ]:
# Single-sensor figure block removed; use combined panel figures below.

In [ ]:
# Plot the first three groups on the same plot
plt.figure(figsize=(10, 6))
# Add dotted vertical lines for specific dates
highlight_dates = [
    (datetime(2007, 6, 1), datetime(2010, 4, 30), 'I. ASCAT A'), 
    (datetime(2010, 5, 1), datetime(2013, 3, 31), 'II. ASCAT A & SMOS'),
    (datetime(2013, 4, 1), datetime(2015, 3, 31), 'III. ASCAT A, B & SMOS'),
    (datetime(2015, 4, 1), datetime(2018, 7, 31), 'IV. ASCAT A, B, SMOS & SMAP'),
    (datetime(2018, 8, 1), datetime(2019, 10, 31), 'V. ASCAT A, B, SMOS, SMAP & CYGNSS'),
    (datetime(2019, 11, 1), datetime(2021, 11, 30), 'VI. ASCAT A, B, C, SMOS, SMAP & CYGNSS'),
    (datetime(2021, 12, 1), datetime(2024, 5, 31), 'VII. ASCAT B, C, SMOS, SMAP & CYGNSS')
]

for i, (start_date, end_date, label) in enumerate(highlight_dates):
    color = 'lightgrey' if i % 2 == 0 else 'darkgrey'
    plt.axvspan(start_date, end_date, color=color, alpha=0.5)
    mid_date = start_date + (end_date - start_date) / 2
    plt.text(mid_date, plt.ylim()[1] * 1.9, label, color='black', ha='center', va='top', fontsize=8, rotation=90)

# Define the first four groups
groups_to_plot = list(species_groups.keys())[:4]

# Plot normalized percent difference for each group
for group in groups_to_plot:
    norm_percent_diff = np.divide(
        (group_ts_DA[group]['OmF_stdv'] - group_ts_OL[group]['OmF_stdv']),
        group_ts_OL[group]['OmF_stdv'],
        out=np.full_like(group_ts_OL[group]['OmF_stdv'], np.nan, dtype=float),
        where=group_ts_OL[group]['OmF_stdv'] != 0
    ) * 100
    
    mean_diff = np.nanmean(norm_percent_diff)
    plt.plot(date_vec_DA, norm_percent_diff, label=f'{group} (Mean: {mean_diff:.3f}%)')


# Set x-ticks using datetime array
plt.xticks(date_vec_DA[::24], rotation=45)

# Customize plot
plt.title('Normalized Percent Difference (OL - DA): LS_DAv8_M36_200006_202405')
plt.xlabel('Date')
plt.ylabel('Normalized Percent Difference (%)')
plt.axhline(y=0, color='black', linestyle=':', linewidth=1)  # Add black dotted line for Y = 0
plt.ylim(-25, 5)
plt.xlim(datetime(2006, 1, 1), datetime(2024, 5, 31))
plt.legend()



plt.tight_layout()
plt.show()

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

panel_groups = ['MODIS', 'ASCAT', 'SMOS', 'SMAP', 'CYGN']
display_name = {'CYGN': 'CYGNSS'}
panel_labels = ['(a)', '(b)', '(c)', '(d)', '(e)']

# Load EASE2 grid and precompute tile -> grid indices
lats2d = np.fromfile(str(EASE_PATH / 'EASE2_M36km.lats.964x406x1.double'), dtype=np.float64).reshape((406, 964))
lons2d = np.fromfile(str(EASE_PATH / 'EASE2_M36km.lons.964x406x1.double'), dtype=np.float64).reshape((406, 964))
lats_row = lats2d[:, 1]
lons_col = lons2d[1, :]
row_idx = np.array([np.abs(lats_row - la).argmin() for la in lat], dtype=int)
col_idx = np.array([np.abs(lons_col - lo).argmin() for lo in lon], dtype=int)


def _grid_from_values(values):
    grid = np.full(lats2d.shape, np.nan, dtype=float)
    vals = np.array(values, dtype=float)
    valid = np.isfinite(vals)
    grid[row_idx[valid], col_idx[valid]] = vals[valid]
    grid = np.where(lats2d < -60.0, np.nan, grid)
    return grid


def _plot_metric_panel(values_by_group, figure_title, units, fixed_range=None, cmap=None):
    fig, axs = plt.subplots(
        3,
        2,
        figsize=(15, 12),
        subplot_kw={'projection': ccrs.Robinson()},
        constrained_layout=True,
    )
    axs = axs.flatten()

    for i, group in enumerate(panel_groups):
        ax = axs[i]
        grid = _grid_from_values(values_by_group[group])

        finite = np.isfinite(grid)
        if fixed_range is not None:
            vmin, vmax = fixed_range
        else:
            if finite.any():
                gmin = float(np.nanmin(grid))
                gmax = float(np.nanmax(grid))
                if gmin < 0:
                    mm = max(abs(gmin), abs(gmax))
                    vmin, vmax = -mm, mm
                else:
                    vmin, vmax = gmin, gmax
            else:
                vmin, vmax = 0.0, 1.0

        if cmap is None:
            cmap_use = 'RdBu_r' if vmin < 0 else 'viridis'
        else:
            cmap_use = cmap

        mesh = ax.pcolormesh(
            lons2d,
            lats2d,
            grid,
            transform=ccrs.PlateCarree(),
            cmap=cmap_use,
            vmin=vmin,
            vmax=vmax,
            shading='auto',
        )

        ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
        ax.coastlines(linewidth=0.5)
        ax.set_global()
        ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

        m = float(np.nanmean(grid))
        s = float(np.nanstd(grid))
        group_name = display_name.get(group, group)
        ax.set_title(f"{panel_labels[i]} {group_name}", fontsize=11)
        ax.text(
            0.02,
            0.03,
            f"Mean: {m:.3f} +/- {s:.3f} {units}",
            transform=ax.transAxes,
            fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'),
        )

        cbar = fig.colorbar(mesh, ax=ax, orientation='horizontal', pad=0.03, fraction=0.05)
        cbar.set_label(units)

    # Keep 3x2 layout with one empty panel
    for j in range(len(panel_groups), len(axs)):
        axs[j].axis('off')

    fig.suptitle(figure_title, fontsize=14)
    plt.show()


values_ol = {g: group_metrics_OL[g]['OmF_stdv'] for g in panel_groups}
values_da = {g: group_metrics_DA[g]['OmF_stdv'] for g in panel_groups}
values_delta = {g: group_metrics_DA[g]['OmF_stdv'] - group_metrics_OL[g]['OmF_stdv'] for g in panel_groups}
values_norm = {
    g: np.divide(
        group_metrics_DA[g]['OmF_stdv'] - group_metrics_OL[g]['OmF_stdv'],
        group_metrics_OL[g]['OmF_stdv'],
        out=np.full_like(group_metrics_OL[g]['OmF_stdv'], np.nan, dtype=float),
        where=group_metrics_OL[g]['OmF_stdv'] != 0,
    ) * 100
    for g in panel_groups
}

_plot_metric_panel(values_ol, 'OL O-F StdDev (3x2 by Sensor)', units='native units')
_plot_metric_panel(values_da, 'DA O-F StdDev (3x2 by Sensor)', units='native units')
_plot_metric_panel(values_delta, 'DA - OL O-F StdDev (3x2 by Sensor)', units='native units')
_plot_metric_panel(values_norm, '(DA - OL) / OL O-F StdDev (%)', units='%', fixed_range=(-60, 60), cmap='RdBu_r')

In [ ]:
# Replaced by combined 2x2 panel map cell above (cell 18).

In [ ]:
# Replaced by combined 2x2 panel map cell above (cell 18).

In [ ]:
# Replaced by combined 2x2 panel map cell above (cell 18).

In [ ]:
# Replaced by combined 2x2 panel map cell above (cell 18).

In [ ]:
# Legacy segmented-period setup (I–VII) aligned with normalized percent-difference timeline
Nmin = 20

period_defs = [
    {
        'label': 'I',
        'start': '20070601',
        'end': '20100430',
        'ol_file': '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/temporal_stats_OL_20070601_20100430.nc4',
    },
    {
        'label': 'II',
        'start': '20100501',
        'end': '20130331',
        'ol_file': '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/temporal_stats_OL_20100501_20130331.nc4',
    },
    {
        'label': 'III',
        'start': '20130401',
        'end': '20150331',
        'ol_file': '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/temporal_stats_OL_20130401_20150331.nc4',
    },
    {
        'label': 'IV',
        'start': '20150401',
        'end': '20180731',
        'ol_file': '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/temporal_stats_OL_20150401_20180731.nc4',
    },
    {
        'label': 'V',
        'start': '20180801',
        'end': '20191031',
        'ol_file': '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/temporal_stats_OL_20180801_20191031.nc4',
    },
    {
        'label': 'VI',
        'start': '20191101',
        'end': '20211130',
        'ol_file': '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/temporal_stats_OL_20191101_20211130.nc4',
    },
    {
        'label': 'VII',
        'start': '20211201',
        'end': '20240531',
        'ol_file': '/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2/temporal_stats_OL_20211201_20240531.nc4',
    },
]


def _compute_group_metrics_from_file(stats_file, nmin=20):
    stats = {}
    with Dataset(stats_file, 'r') as nc:
        for key, value in nc.variables.items():
            stats[key] = value[:].filled(np.nan)

    N_data = stats['N_data'].copy()
    OmF_mean = stats['OmF_mean'].copy()
    OmF_stdv = stats['OmF_stdv'].copy()
    OmF_norm_mean = stats['OmF_norm_mean'].copy()
    OmF_norm_stdv = stats['OmF_norm_stdv'].copy()
    OmA_mean = stats['OmA_mean'].copy()
    OmA_stdv = stats['OmA_stdv'].copy()

    OmF_mean[N_data < nmin] = np.nan
    OmF_stdv[N_data < nmin] = np.nan
    OmF_norm_mean[N_data < nmin] = np.nan
    OmF_norm_stdv[N_data < nmin] = np.nan
    OmA_mean[N_data < nmin] = np.nan
    OmA_stdv[N_data < nmin] = np.nan
    N_data[N_data < nmin] = 0

    group_metrics = {}
    for group, species_indices in species_groups.items():
        group_metrics[group] = {}

        weights = N_data[:, species_indices]
        group_N_data = np.nansum(weights, axis=1)

        def _wavg(arr):
            num = np.nansum(arr[:, species_indices] * weights, axis=1)
            return np.divide(
                num,
                group_N_data,
                out=np.full(group_N_data.shape, np.nan, dtype=float),
                where=group_N_data > 0,
            )

        group_metrics[group]['OmF_mean'] = _wavg(OmF_mean)
        group_metrics[group]['OmF_stdv'] = _wavg(OmF_stdv)
        group_metrics[group]['OmF_norm_mean'] = _wavg(OmF_norm_mean)
        group_metrics[group]['OmF_norm_stdv'] = _wavg(OmF_norm_stdv)
        group_metrics[group]['OmA_mean'] = _wavg(OmA_mean)
        group_metrics[group]['OmA_stdv'] = _wavg(OmA_stdv)
        group_metrics[group]['Nobs_data'] = group_N_data

    return group_metrics


# Build OL segmented metrics (7 periods)
group_metrics_OL_list = []
for p in period_defs:
    print(f"reading OL stats nc4 file {p['ol_file']}")
    group_metrics_OL_list.append(_compute_group_metrics_from_file(p['ol_file'], nmin=Nmin))

(
    group_metrics_OL_1,
    group_metrics_OL_2,
    group_metrics_OL_3,
    group_metrics_OL_4,
    group_metrics_OL_5,
    group_metrics_OL_6,
    group_metrics_OL_7,
) = group_metrics_OL_list

In [ ]:
# Build DA segmented metrics (7 periods) aligned to period_defs
for p in period_defs:
    p['da_file'] = p['ol_file'].replace('temporal_stats_OL_', 'temporal_stats_DA_')

group_metrics_DA_list = []
for p in period_defs:
    print(f"reading DA stats nc4 file {p['da_file']}")
    group_metrics_DA_list.append(_compute_group_metrics_from_file(p['da_file'], nmin=Nmin))

(
    group_metrics_DA_1,
    group_metrics_DA_2,
    group_metrics_DA_3,
    group_metrics_DA_4,
    group_metrics_DA_5,
    group_metrics_DA_6,
    group_metrics_DA_7,
) = group_metrics_DA_list

In [ ]:
def _plot_sensor_period_maps(group_name, first_period_label):
    period_order = [p['label'] for p in period_defs]
    start_idx = period_order.index(first_period_label)

    for i in range(start_idx, len(period_defs)):
        p = period_defs[i]
        ol_vals = group_metrics_OL_list[i][group_name]['OmF_stdv']
        da_vals = group_metrics_DA_list[i][group_name]['OmF_stdv']

        map_array[:, 0] = np.divide(
            da_vals - ol_vals,
            ol_vals,
            out=np.full_like(ol_vals, np.nan, dtype=float),
            where=ol_vals != 0,
        ) * 100

        maxval = np.nanmax(map_array[:, 0])
        minval = np.nanmin(map_array[:, 0])

        plot_global_tight_pcm(
            map_array,
            False,
            True,
            f"(DA - OL) / OL OmF StdDev {group_name} (Time period {p['label']}: {p['start']}-{p['end']})\n (Max: {maxval:.3g} Min: {minval:.3g})",
            '%',
            -60,
            60,
        )


_plot_sensor_period_maps('ASCAT', 'I')

In [ ]:
_plot_sensor_period_maps('SMOS', 'II')

In [ ]:
_plot_sensor_period_maps('SMAP', 'IV')

In [ ]:
_plot_sensor_period_maps('CYGN', 'V')

In [ ]:
# Postage-stamp matrix: normalized O-F StdDev change by period (rows) and sensor (columns)
import cartopy.crs as ccrs

required = ['period_defs', 'group_metrics_OL_list', 'group_metrics_DA_list', 'lat', 'lon']
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Run the segmented-period setup cells first. Missing: {missing}")

sensor_cols = [('ASCAT', 'ASCAT'), ('SMOS', 'SMOS'), ('SMAP', 'SMAP'), ('CYGN', 'CYGNSS')]

# EASE2 M36 grid
lats2d = np.fromfile(str(EASE_PATH / 'EASE2_M36km.lats.964x406x1.double'), dtype=np.float64).reshape((406, 964))
lons2d = np.fromfile(str(EASE_PATH / 'EASE2_M36km.lons.964x406x1.double'), dtype=np.float64).reshape((406, 964))
lats_row = lats2d[:, 1]
lons_col = lons2d[1, :]

# Tile -> grid index map
row_idx = np.array([np.abs(lats_row - la).argmin() for la in lat], dtype=int)
col_idx = np.array([np.abs(lons_col - lo).argmin() for lo in lon], dtype=int)


def _to_grid(values):
    grid = np.full(lats2d.shape, np.nan, dtype=float)
    vals = np.array(values, dtype=float)
    valid = np.isfinite(vals)
    grid[row_idx[valid], col_idx[valid]] = vals[valid]
    # Match other map sections: exclude Antarctica
    grid = np.where(lats2d < -60.0, np.nan, grid)
    return grid


nrows = len(period_defs)
ncols = len(sensor_cols)
fig, axs = plt.subplots(
    nrows,
    ncols,
    figsize=(11.5, 14.5),
    subplot_kw={'projection': ccrs.Robinson()},
)

# Ensure 2D indexing when nrows/ncols are 1
axs = np.atleast_2d(axs)

mesh = None
for r, p in enumerate(period_defs):
    for c, (group, disp_name) in enumerate(sensor_cols):
        ax = axs[r, c]

        ol_vals = group_metrics_OL_list[r][group]['OmF_stdv']
        da_vals = group_metrics_DA_list[r][group]['OmF_stdv']
        rel = np.divide(
            da_vals - ol_vals,
            ol_vals,
            out=np.full_like(ol_vals, np.nan, dtype=float),
            where=ol_vals != 0,
        ) * 100

        grid = _to_grid(rel)
        finite = np.isfinite(grid)

        mesh = ax.pcolormesh(
            lons2d,
            lats2d,
            grid,
            transform=ccrs.PlateCarree(),
            cmap='RdBu_r',
            vmin=-60,
            vmax=60,
            shading='auto',
        )
        ax.coastlines(linewidth=0.25)
        ax.set_global()
        ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

        # Column headers on top row
        if r == 0:
            ax.set_title(disp_name, fontsize=10, pad=2)

        # Row labels on first column
        if c == 0:
            ax.text(
                -0.08,
                0.5,
                f"{p['label']}",
                transform=ax.transAxes,
                va='center',
                ha='right',
                fontsize=9,
                fontweight='bold',
            )

        # Mean ± std below each panel
        if finite.any():
            mean_val = float(np.nanmean(grid))
            std_val = float(np.nanstd(grid))
            stat_txt = f"{mean_val:.1f} +/- {std_val:.1f}%"
        else:
            stat_txt = 'N/A'
            ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes, ha='center', va='center', fontsize=8)

        ax.text(
            0.5,
            -0.03,
            stat_txt,
            transform=ax.transAxes,
            ha='center',
            va='top',
            fontsize=6,
            clip_on=False,
        )

fig.suptitle('(DA - OL) / OL O-F StdDev (%) by Period and Sensor', fontsize=12, y=0.995)

# Tight postage-stamp spacing
plt.subplots_adjust(left=0.08, right=0.995, top=0.965, bottom=0.055, wspace=0.01, hspace=0.0)

cbar = fig.colorbar(mesh, ax=axs.ravel().tolist(), orientation='horizontal', fraction=0.02, pad=0.02)
cbar.set_label('(DA - OL) / OL OmF StdDev (%)', fontsize=10)
cbar.set_ticks([-60, -30, 0, 30, 60])

plt.show()

In [ ]:
import matplotlib as mpl
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt

pres_rc = {
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 17,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 14,
    "lines.linewidth": 2.2,
    "axes.linewidth": 1.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
}

epochs = [
    (datetime(2000, 6, 1),  datetime(2007, 5, 31),  'MODIS'),
    (datetime(2007, 6, 1),  datetime(2010, 4, 30),  'MODIS\n+ ASCAT'),
    (datetime(2010, 5, 1),  datetime(2015, 3, 31),  'MODIS + ASCAT\n+ SMOS'),
    (datetime(2015, 4, 1),  datetime(2018, 7, 31),  'MODIS + ASCAT\n+ SMOS + SMAP'),
    (datetime(2018, 8, 1),  datetime(2024, 5, 31),  'MODIS + ASCAT\n+ SMOS + SMAP\n+ CYGNSS'),
]

groups_to_plot = ["MODIS", "SMOS", "SMAP", "ASCAT", "CYGN"]
display_names   = {"CYGN": "CYGNSS"}
colors = {"MODIS": "#9467bd", "SMOS": "#1f77b4", "SMAP": "#ff7f0e", "ASCAT": "#2ca02c", "CYGN": "#d62728"}

with mpl.rc_context(pres_rc):
    fig, ax = plt.subplots(figsize=(14, 6), layout="constrained")

    # Shaded epoch bands
    for i, (start, end, label) in enumerate(epochs):
        color = "#e8e8e8" if i % 2 == 0 else "#c8c8c8"
        ax.axvspan(start, end, color=color, alpha=0.7, zorder=0)
        mid = start + (end - start) / 2
        ax.text(mid, 4.2, label, color="black", ha="center", va="top",
                fontsize=11, rotation=0, linespacing=1.3)

    # Plot lines
    for group in groups_to_plot:
        norm_pct = np.divide(
            (group_ts_DA[group]["OmF_stdv"] - group_ts_OL[group]["OmF_stdv"]),
            group_ts_OL[group]["OmF_stdv"],
            out=np.full_like(group_ts_OL[group]["OmF_stdv"], np.nan, dtype=float),
            where=group_ts_OL[group]["OmF_stdv"] != 0
        ) * 100
        label = display_names.get(group, group)
        mean_diff = np.nanmean(norm_pct)
        ax.plot(date_vec_DA, norm_pct,
                label=f"{label} (mean: $\mathbf{{{mean_diff:.1f}\%}}$)",
                color=colors[group], zorder=3,
                linewidth=1.0 if group == "MODIS" else 2.2)

    ax.axhline(y=0, color="black", linestyle=":", linewidth=1.2, zorder=2)
    ax.set_ylim(-30, 5)
    ax.set_xlim(datetime(2000, 6, 1), datetime(2024, 5, 31))
    ax.set_xlabel("Date")
    ax.set_ylabel("Normalized Percent Difference (%)")
    ax.set_xticks(date_vec_DA[::24])
    ax.tick_params(axis="x", rotation=45)
    ax.legend(loc="lower left", frameon=True, facecolor="white",
              edgecolor="gray", framealpha=0.85)
    ax.grid(axis="y", alpha=0.35, linewidth=0.8)

    plt.savefig("norm_pct_diff_presentation.png", dpi=200,
                bbox_inches="tight", pad_inches=0.15, transparent=True)
    plt.show()
